In [21]:
import sys
from pathlib import Path

import torch
from mlflow.pyfunc.stdin_server import model
from torch.nn import Sequential, Conv2d, ReLU, Module, MaxPool2d, Linear, Flatten
import torch.optim as optim
from torch.nn.functional import relu

from src.data.dataset import DataLoader
from src.data.dataset import create_data_loaders

In [10]:
PROJECT_ROOT = Path.cwd().parent
SRC_DIR = PROJECT_ROOT / "src"

if str(SRC_DIR) not in sys.path:
    sys.path.append(str(SRC_DIR))

In [11]:
train_loader, validation_loader, test_loader = create_data_loaders()

In [12]:
images, labels = next(iter(train_loader))

print(images.shape)
print(labels.shape)
print(train_loader.dataset.class_to_idx)

torch.Size([32, 3, 128, 128])
torch.Size([32])
{'cats': 0, 'dogs': 1}


In [15]:
conv_1 = Conv2d(
    in_channels=3,
    out_channels=32,
    kernel_size=3,
    padding=1
)

conv_1_output = conv_1(images)
conv_1_output.shape

torch.Size([32, 32, 128, 128])

32  - batch size

32  - feature maps

128 - height

128 - width

In [16]:
relu = ReLU()

relu_output = relu(conv_1_output)

print(relu_output.shape)

torch.Size([32, 32, 128, 128])


In [18]:
pool = MaxPool2d(kernel_size=2)

pool_output = pool(relu_output)

print(pool_output.shape)

torch.Size([32, 32, 64, 64])


In [19]:
def conv_block_x(imgs):
    conv = Conv2d(
        in_channels=3,
        out_channels=32,
        kernel_size=3,
        padding=1
    )

    relu_x = ReLU()
    pool_x = MaxPool2d(kernel_size=2)

    output = conv(imgs)
    print(output.shape)

    output = relu_x(output)
    print(output.shape)

    output = pool_x(output)
    print(output.shape)

    return output

In [20]:
conv_block_x(images)

torch.Size([32, 32, 128, 128])
torch.Size([32, 32, 128, 128])
torch.Size([32, 32, 64, 64])


tensor([[[[0.6304, 0.6304, 0.6304,  ..., 0.6304, 0.6304, 0.6304],
          [0.6304, 0.6304, 0.6304,  ..., 0.6304, 0.6304, 0.6304],
          [0.6304, 0.6304, 0.6304,  ..., 0.6304, 0.6304, 0.6304],
          ...,
          [0.6304, 0.6304, 0.6304,  ..., 0.6304, 0.6304, 0.6304],
          [0.6304, 0.6304, 0.6304,  ..., 0.6304, 0.6304, 0.6304],
          [0.6304, 0.6304, 0.6304,  ..., 0.6304, 0.6304, 0.6304]],

         [[0.1587, 0.1587, 0.1587,  ..., 0.1587, 0.1587, 0.1587],
          [0.0000, 0.0000, 0.0000,  ..., 0.0000, 0.0000, 0.0000],
          [0.0000, 0.0000, 0.0000,  ..., 0.0000, 0.0000, 0.0000],
          ...,
          [0.0000, 0.0000, 0.0000,  ..., 0.0000, 0.0000, 0.0000],
          [0.0000, 0.0000, 0.0000,  ..., 0.0000, 0.0000, 0.0000],
          [0.1739, 0.0000, 0.0000,  ..., 0.0000, 0.0000, 0.0000]],

         [[0.4317, 0.4317, 0.4317,  ..., 0.4317, 0.4317, 0.4317],
          [0.4317, 0.4317, 0.4317,  ..., 0.4317, 0.4317, 0.4317],
          [0.4317, 0.4317, 0.4317,  ..., 0

In [22]:
class BaseCNN(Module):

    def __init__(self):
        super().__init__()

        self.features = Sequential(
            Conv2d(in_channels=3, out_channels=32, kernel_size=3, padding=1),
            ReLU(),
            MaxPool2d(kernel_size=2),
            Conv2d(in_channels=32, out_channels=64, kernel_size=3, padding=1),
            ReLU(),
            MaxPool2d(kernel_size=2),
        )

        self.classifier = Sequential(
            Flatten(),
            Linear(in_features=64 * 32 * 32, out_features=128),
            ReLU(),
            Linear(in_features=128, out_features=2),
        )

    def forward(self, x):
        x = self.features(x)
        x = self.classifier(x)

        return x

In [23]:
model = BaseCNN()

In [24]:
print(model)

BaseCNN(
  (features): Sequential(
    (0): Conv2d(3, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): ReLU()
    (2): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (3): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (4): ReLU()
    (5): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  )
  (classifier): Sequential(
    (0): Flatten(start_dim=1, end_dim=-1)
    (1): Linear(in_features=65536, out_features=128, bias=True)
    (2): ReLU()
    (3): Linear(in_features=128, out_features=2, bias=True)
  )
)


In [25]:
images, labels = next(iter(train_loader))

outputs = model(images)

print(f"Input shape: {images.shape}")
print(f"Output shape: {outputs.shape}")

Input shape: torch.Size([32, 3, 128, 128])
Output shape: torch.Size([32, 2])


In [26]:
outputs = model(images)

In [27]:
print(outputs)

tensor([[-0.0283,  0.0412],
        [-0.0001,  0.0485],
        [-0.0173,  0.0512],
        [-0.0042,  0.0570],
        [-0.0023,  0.0390],
        [-0.0066,  0.0588],
        [-0.0397,  0.0758],
        [-0.0132,  0.0416],
        [-0.0241,  0.0514],
        [-0.0121,  0.0407],
        [-0.0221,  0.0645],
        [ 0.0091,  0.0348],
        [-0.0184,  0.0630],
        [-0.0252,  0.0302],
        [-0.0271,  0.0441],
        [ 0.0110,  0.0256],
        [-0.0131,  0.0572],
        [-0.0306,  0.0605],
        [-0.0066,  0.0430],
        [-0.0159,  0.0191],
        [ 0.0211,  0.0275],
        [-0.0196,  0.0327],
        [-0.0122,  0.0523],
        [ 0.0080,  0.0273],
        [-0.0250,  0.0614],
        [-0.0183,  0.0606],
        [-0.0299,  0.0722],
        [-0.0192,  0.0578],
        [-0.0188,  0.0380],
        [-0.0238,  0.0452],
        [-0.0183,  0.0440],
        [-0.0230,  0.0546]], grad_fn=<AddmmBackward0>)


In [28]:
predictions = torch.argmax(outputs, dim=1)
print(predictions)

tensor([1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
        1, 1, 1, 1, 1, 1, 1, 1])


In [31]:
print(train_loader.dataset.class_to_idx)

{'cats': 0, 'dogs': 1}
